# 06 YOLOv5 的目标分配：一个真实框可以教多个预测

上一课理解了 anchor 是参考宽高模板。这一课继续回答：**训练图片中有一个真实人脸框，究竟让哪些 anchor、哪些网格位置来学习它？**

先记住本地 YOLOv5 的关键区别：它不是在同一网格的候选预测中，只选当前 IoU 最大的一个负责；它可以为一个真实目标建立多个正样本监督。

## 1. 为什么仍然需要分配规则

每张图片只有有限个人工标注目标，但三个检测尺度上有许多预测位置，每个位置还对应三个 anchor。

训练程序必须明确：哪些预测负责学习某个人脸，哪些没有获得真实目标的分配。

这张训练责任表由程序根据真实标注和匹配规则构造。模型不需要先预测出高置信度，才有资格获得定位训练。

以下例子只演示一个目标、一个尺度，不代表完整训练中只处理这个尺度。

## 2. 第一步：检查参考宽高是否合适

设输入图片经过训练预处理后，某个人脸的真实框宽 32、高 40 像素。某个尺度的三个参考模板如下；这些数值是教学例子，不是当前权重的实际 anchor。

| Anchor | 参考宽高 | 与真实框的尺寸关系 |
|---|---|---|
| A | 20 × 30 | 比真实框小一些 |
| B | 40 × 20 | 宽度接近，高度小一些 |
| C | 160 × 200 | 明显大很多 |

本地代码比较真实框与 anchor 的宽高比例，而不是在这里比较当前预测框的 IoU。

为了同时衡量“模板太大”和“模板太小”，取以下四个比值中的最大值：

$$
m=\max\left(\frac{w}{a_w},\frac{a_w}{w},\frac{h}{a_h},\frac{a_h}{h}\right)
$$

其中 $w,h$ 为真实框宽高，$a_w,a_h$ 为参考宽高。$m$ 越接近 1，尺寸越接近。

正式代码会先把真实框与 anchor 都换成对应尺度的网格单位；本例用相同的像素单位计算，比例相同。

## 3. 不是只取尺寸最接近的那一个

设本例采用仓库示例超参数中的匹配阈值 4，要求 $m<4$。

| Anchor | 最大比例 m | 是否匹配 |
|---|---:|---|
| A | 1.6 | 是 |
| B | 2.0 | 是 |
| C | 5.0 | 否 |

A 与 B 都满足条件，所以都可以为这个真实目标提供正样本预测。不会因为 A 更接近，就只留下 A。

C 没匹配这个目标，表示它没有获得针对该目标的定位与类别监督；若有其他目标，它仍可能与其他目标匹配。

完整流程会在各检测尺度上分别执行检查。一个真实目标可能匹配多个 anchor，甚至不止一个尺度。阈值取决于实际训练超参数，本课没有启动训练。

## 4. 第二步：根据真实中心找到网格

设使用步长为 8 的 P3 尺度，真实框中心在输入图的像素坐标 $(82,122)$。

换成该尺度的网格坐标：

$$
x_g=82/8=10.25,\qquad y_g=122/8=15.25
$$

因此，中心落在编号为“第 10 列、第 15 行”的网格中。这里列、行都从 0 开始。

这个编号表示位置，不是类别，也不是 anchor 编号。A、B 两种匹配的 anchor，都可以在这个位置获得针对该真实框的监督。

## 5. 第三步：加入符合条件的邻近位置

当前代码还检查中心在网格内部的位置，并在符合条件且未越界时，为部分相邻网格增加监督。

在本例中，中心的格内位置是 $(0.25,0.25)$，处于该网格靠左、靠上的一侧。因此，除了自身位置，还会加入左邻和上邻位置。

以下始终使用“列、行”的顺序：

| 位置 | 列、行编号 |
|---|---|
| 中心所在网格 | (10, 15) |
| 左邻网格 | (9, 15) |
| 上邻网格 | (10, 14) |

本例没有加入左上角的对角网格，也不是把所有周围网格一律加入。

相邻网格对应的预测并不需要把框中心锁在自己的格子内；当前中心解码允许一定范围的跨格偏移。这样可以让网格边界附近的目标获得更多监督机会。

## 6. 一个真实框，在本例产生六条正样本对应

两种匹配 anchor，分别在三个位置获得监督：

| Anchor | 网格位置（列、行） | 要学习的真实目标 |
|---|---|---|
| A | (10, 15) | 同一张人脸 |
| A | (9, 15) | 同一张人脸 |
| A | (10, 14) | 同一张人脸 |
| B | (10, 15) | 同一张人脸 |
| B | (9, 15) | 同一张人脸 |
| B | (10, 14) | 同一张人脸 |

因此，本例在这一个尺度上有六条正样本对应关系，不是图片里有六张真实人脸。

六条对应学习的是同一个真实框，但框中心需要相对于各自的网格位置表示，所以具体的局部坐标目标并不完全相同。

正样本身份只表示“你负责学习这个目标”，不表示模型目前已经预测正确。

## 7. 分配之后，损失才知道怎样监督

获得分配的预测，会参与相应的定位、类别和目标存在性监督。没有获得目标分配的位置，不会凭空获得一个需要回归的真实框，但仍可能参与目标存在性的负样本监督。

本地 YOLOv5 的目标存在性监督还会利用定位质量来构造目标值，不能把所有正样本都简单解释为“置信度填 1”。具体数值如何生成，放到损失函数一课讨论。

多个预测可以学习同一个目标，因此推理时也可能出现重复框。推理阶段不再依据真实标注分配责任，而是用分数筛选和 NMS 等后处理得到结果。

目标分配发生在训练中；NMS 是推理后处理。二者解决不同问题。

## 8. 自检

1. A、B 都通过宽高匹配阈值，是否只保留更接近真实框的 A？
2. 同一个真实框得到六条正样本对应，是否表示真实图片中有六个人脸？
3. 一个框当前预测很差，是否就不能成为正样本？
4. 对一张没有真实标注的新图片做推理时，还能否按上述规则分配真实目标？

<details>
<summary>参考答案</summary>

1. 不是。当前规则保留所有通过条件的匹配。
2. 不是。同一个真实目标可以监督多个预测位置。
3. 不是。正样本是分配到的训练责任，不等于当前预测正确；本节匹配规则不依赖当前预测置信度。
4. 不能。没有真实框就不能构造这样的监督分配；推理使用预测与后处理。

</details>

下一课：分配后的预测如何计算定位、目标存在性和类别三部分损失。

## 本课依据与边界

- 本地 `vendor/yolov5/utils/loss.py` 的 `ComputeLoss.build_targets`：宽高比例匹配、网格坐标换算和邻近位置分配。
- 本地 `vendor/yolov5/data/hyps/hyp.scratch-low.yaml`：示例 `anchor_t: 4.0`。
- 本地固定源码版本：`af08ba156fd58ed5b0fe7e4e58a22c72c3cdd0ce`。

本例只演示单个目标在一个尺度上的分配，不展开多个目标竞争同一位置等复杂情况，也不代表所有 YOLO 版本使用同一规则。宽高和中心均为教学数值，算例已检查。

本课仅新增学习笔记，没有运行正式数据划分、改变超参数或启动训练。